# 07 Genie Agent: benchmark the space through the Genie Conversation API

Asks the Genie Space **Blue Ridge LNG — PM Co-Pilot (synthetic)** fifteen questions, each as a **fresh conversation**, and scores every answer against a verified SQL result. All data is synthetic.

* **10 curated questions** (`sample_questions.md`): the same ten are loaded into the space as example SQL, so this measures whether Genie reproduces its curated answers through the live API.
* **5 held-out questions** (`heldout_questions.md`): phrased differently, covering other angles, and **never loaded into the space** (notebook `01_create_space` asserts this). The space was not tuned toward them.
* **Pass bar:** at least 8 of 10 curated correct and at least 3 of 5 held-out correct.

**How a question is asked.** The Genie Conversation API over REST (`POST .../start-conversation`, poll `GET .../messages/{id}` until `COMPLETED`, then `GET .../attachments/{id}/query-result`). The serverless SDK in this runtime has no `w.genie` yet, so the calls go through `WorkspaceClient.api_client`; the endpoints are the same ones the SDK wraps. The notebook runs as the submitting user, which is the same shape as the app's on-behalf-of call (the signed-in PM's own permissions).

**Captured per question:** Genie's text answer, the generated SQL, the result rows (the first five are shown, but scoring uses every returned row), the expected answer, and a verdict.

**Scoring (deterministic, no LLM judge).** Each question carries a list of *key facts* from the verified SQL result (numbers, IDs, labels), listed with the question's rule in `benchmark_expected.json`. A fact is found if it appears in Genie's text answer or in any result row: numbers are compared numerically (commas, `$`, `M`/`million` suffixes parsed; a percent may appear as a 0-1 fraction when the rule says so); text is a case-insensitive substring.
* **correct**: every key fact is found, and no cap below applies.
* **partial**: at least half of the key facts are found, **or** all are found but a cap applies: a value the rule forbids is in the result rows (for example a due-today milestone in the "late" list), the generated SQL does not read the table the rule requires, or the answer presents `slip_probability` as a percentage or a chance.
* **wrong**: fewer than half of the key facts, an empty result, no SQL, or an error.

A probability shown as a percentage in any answer is recorded as a finding in the transcript.

**Genie is not deterministic.** The same question can produce different SQL or wording on another run; this is one run. No question is retried. The slowest step is Genie itself, so each question waits up to 4 minutes.

In [1]:
import json, os, re, time, datetime
from databricks.sdk import WorkspaceClient

dbutils.widgets.text("run_id", "", "Job run id")
RUN_ID = dbutils.widgets.get("run_id")
print("job run id:", RUN_ID or "(interactive)")

w = WorkspaceClient()
print("running as:", w.current_user.me().user_name)
SPACE_ID = open("outputs/space_id.txt").read().strip()
print("space id:", SPACE_ID)
space = w.api_client.do("GET", f"/api/2.0/genie/spaces/{SPACE_ID}")
print("space title:", space["title"])

bench = json.load(open("benchmark_expected.json"))
QUESTIONS = [("curated", q) for q in bench["curated"]] + [("heldout", q) for q in bench["heldout"]]
print(len(bench["curated"]), "curated +", len(bench["heldout"]), "held-out questions; as-of", bench["as_of_date"])

job run id: 770421906253489
running as: joseph.hobbs@databricks.com
space id: 01f1db9edffa1a298195db2806ce45e3
space title: Blue Ridge LNG — PM Co-Pilot (synthetic)
10 curated + 5 held-out questions; as-of 2026-11-06


## 1. Ask and capture
One fresh conversation per question: question in, then the text answer, generated SQL and result rows out.

In [2]:
BASE = f"/api/2.0/genie/spaces/{SPACE_ID}"

def ask(question, timeout_s=240):
    t0 = time.time()
    out = {"question": question, "status": None, "error": None, "text": "", "sql": None, "columns": [], "rows": [], "seconds": None}
    try:
        started = w.api_client.do("POST", f"{BASE}/start-conversation", body={"content": question})
        cid, mid = started["conversation_id"], started["message_id"]
        out["conversation_id"], out["message_id"] = cid, mid
        while True:
            msg = w.api_client.do("GET", f"{BASE}/conversations/{cid}/messages/{mid}")
            if msg["status"] in ("COMPLETED", "FAILED", "CANCELLED"):
                break
            if time.time() - t0 > timeout_s:
                out["error"] = "timeout waiting for Genie"
                break
            time.sleep(3)
        out["status"] = msg["status"]
        if msg.get("error"):
            out["error"] = json.dumps(msg["error"])[:500]
        texts = []
        for a in msg.get("attachments", []) or []:
            if "text" in a and a["text"].get("content"):
                texts.append(a["text"]["content"].strip())
            if "query" in a and a["query"].get("query") and out["sql"] is None:
                out["sql"] = a["query"]["query"].strip()
                if a["query"].get("description"):
                    texts.insert(0, a["query"]["description"].strip())
                res = w.api_client.do("GET", f"{BASE}/conversations/{cid}/messages/{mid}/attachments/{a['attachment_id']}/query-result")
                sr = res.get("statement_response", {})
                out["columns"] = [c["name"] for c in sr.get("manifest", {}).get("schema", {}).get("columns", [])]
                out["rows"] = sr.get("result", {}).get("data_array", []) or []
        out["text"] = "\n\n".join(dict.fromkeys(texts))
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {e}"[:500]
    out["seconds"] = round(time.time() - t0, 1)
    return out

results = []
for kind, q in QUESTIONS:
    r = ask(q["question"])
    r.update(kind=kind, n=q["n"])
    results.append(r)
    print(f"[{kind} {q['n']:>2}] {r['status']} {r['seconds']:>5}s rows={len(r['rows'])}  {q['question']}")

[curated  1] COMPLETED  13.0s rows=1  How many drawings failed compliance review this week, out of how many reviewed?
[curated  2] COMPLETED  13.1s rows=5  Show failed compliance findings this week by code citation.
[curated  3] COMPLETED  10.1s rows=5  Which piping isometric drawings failed compliance this week?
[curated  4] COMPLETED  13.0s rows=6  Which milestones are already late, and by how many days?
[curated  5] COMPLETED  16.1s rows=3  What are the top 3 highest-risk milestones in the next 90 days?
[curated  6] COMPLETED  10.2s rows=1  Explain the budget variance: how far over budget are we?
[curated  7] COMPLETED  10.2s rows=2  How many open NCRs do we have, by category?
[curated  8] COMPLETED  13.0s rows=1  What is our safety record: TRIR and days without an incident?
[curated  9] COMPLETED  10.0s rows=1  What does OSHA 1910.147 require?
[curated 10] COMPLETED   9.9s rows=1  How accurate is the AI compliance judge?
[heldout  1] COMPLETED  13.0s rows=1  Which discipline has th

## 2. Score
Deterministic rules described above.

In [3]:
MULT = {"k": 1e3, "thousand": 1e3, "m": 1e6, "million": 1e6, "b": 1e9, "billion": 1e9}
NUM = re.compile(r"(?<![A-Za-z0-9.])-?\d+(?:,\d{3})*(?:\.\d+)?(?:[eE][+-]?\d+)?(?:\s*(?:million|billion|thousand|[MBKkmb])\b)?")

def numbers(text):
    vals = []
    for m in NUM.finditer(text or ""):
        s = m.group(0).strip()
        mult = 1.0
        suffix = re.search(r"(million|billion|thousand|[MBKkmb])$", s)
        if suffix and not re.search(r"[eE][+-]?\d+$", s):
            mult = MULT[suffix.group(1).lower()]
            s = s[: suffix.start()].strip()
        try:
            vals.append(float(s.replace(",", "")) * mult)
        except ValueError:
            pass
    return vals

def has_number(vals, n, pct):
    for v in vals:
        cands = [v, v * 100] if pct else [v]
        for c in cands:
            if isinstance(n, int) and not pct:
                if abs(c - n) < 1e-9:
                    return True
            elif pct:
                if abs(c - n) <= 0.5:
                    return True
            elif abs(c - n) <= max(0.051, 0.005 * abs(n)):
                return True
    return False

SLIP_PCT = re.compile(r"slip\w*[^.\n]{0,80}\d+(?:\.\d+)?\s*%|\d+(?:\.\d+)?\s*%[^.\n]{0,80}slip|\d+(?:\.\d+)?\s*%\s*(?:chance|probability|likel)|(?:chance|probability|likelihood)[^.\n]{0,60}\d+(?:\.\d+)?\s*%", re.I)

def score(res, exp):
    rule = exp["rule"]
    rows_text = " ".join(str(c) for row in res["rows"] for c in row)
    hay = (res["text"] or "") + " " + rows_text
    vals = numbers(hay)
    found, missing = [], []
    for t in rule["tokens"]:
        ok = has_number(vals, t, rule.get("pct", False)) if isinstance(t, (int, float)) else (str(t).casefold() in hay.casefold())
        (found if ok else missing).append(t)
    notes = []
    if res["error"] or res["status"] != "COMPLETED" or not res["sql"] or not res["rows"]:
        return "wrong", found, missing, ["error, no SQL or empty result: " + (res["error"] or f"status={res['status']}, sql={'yes' if res['sql'] else 'no'}, rows={len(res['rows'])}")], False
    frac = len(found) / len(rule["tokens"])
    cap = False
    for f in rule.get("forbid_in_rows", []):
        if f.casefold() in rows_text.casefold():
            cap = True; notes.append(f"forbidden value in result rows: {f}")
    if rule.get("sql_contains") and rule["sql_contains"].casefold() not in res["sql"].casefold():
        cap = True; notes.append(f"generated SQL does not read {rule['sql_contains']}")
    slip = bool(SLIP_PCT.search(res["text"] or ""))
    if slip:
        cap = True; notes.append("FINDING: answer presents slip_probability as a percentage or chance")
    if "slip_probability" in res["sql"].lower():
        notes.append("note: generated SQL selects slip_probability")
    if frac == 1.0 and not cap:
        v = "correct"
    elif frac >= 0.5:
        v = "partial"
    else:
        v = "wrong"
    return v, found, missing, notes, slip

for res, (kind, q) in zip(results, QUESTIONS):
    v, found, missing, notes, slip = score(res, q)
    res.update(verdict=v, found=found, missing=missing, notes=notes, slip_pct_finding=slip,
               expected_answer=q["expected_answer"], expected_rows=q["expected_rows"], rule_desc=q["rule"]["desc"])
    print(f"[{kind} {q['n']:>2}] {v.upper():8} found {len(found)}/{len(q['rule']['tokens'])}  missing={missing}  {'; '.join(notes)}")

def tally(kind):
    rs = [r for r in results if r["kind"] == kind]
    return {v: sum(1 for r in rs if r["verdict"] == v) for v in ("correct", "partial", "wrong")}, len(rs)
(cur, nc), (held, nh) = tally("curated"), tally("heldout")
print()
print(f"CURATED : {cur['correct']}/{nc} correct, {cur['partial']} partial, {cur['wrong']} wrong   (pass bar 8/10)")
print(f"HELD-OUT: {held['correct']}/{nh} correct, {held['partial']} partial, {held['wrong']} wrong   (pass bar 3/5)")
print("PASS" if cur["correct"] >= 8 and held["correct"] >= 3 else "BELOW BAR")
print("slip_probability-as-percentage findings:", sum(1 for r in results if r["slip_pct_finding"]))

[curated  1] CORRECT  found 2/2  missing=[]  
[curated  2] CORRECT  found 10/10  missing=[]  
[curated  3] CORRECT  found 5/5  missing=[]  
[curated  4] CORRECT  found 11/11  missing=[]  
[curated  5] CORRECT  found 3/3  missing=[]  
[curated  6] CORRECT  found 3/3  missing=[]  
[curated  7] CORRECT  found 4/4  missing=[]  
[curated  8] CORRECT  found 2/2  missing=[]  
[curated  9] CORRECT  found 2/2  missing=[]  
[curated 10] CORRECT  found 2/2  missing=[]  
[heldout  1] CORRECT  found 2/2  missing=[]  
[heldout  2] CORRECT  found 3/3  missing=[]  
[heldout  3] CORRECT  found 2/2  missing=[]  
[heldout  4] CORRECT  found 2/2  missing=[]  
[heldout  5] CORRECT  found 1/1  missing=[]  

CURATED : 10/10 correct, 0 partial, 0 wrong   (pass bar 8/10)
HELD-OUT: 5/5 correct, 0 partial, 0 wrong   (pass bar 3/5)
PASS
slip_probability-as-percentage findings: 0


## 3. Write the transcript
Writes `outputs/genie_qa_results.json` (raw capture) and `outputs/genie_qa_transcript.md` (copied to `evidence/genie_qa_transcript.md`), and prints the transcript.

In [4]:
def table(columns, rows, limit=5):
    if not rows:
        return "(no rows)"
    cols = columns or [f"col{i}" for i in range(len(rows[0]))]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for row in rows[:limit]:
        lines.append("| " + " | ".join(str(c).replace("|", "\\|").replace("\n", " ")[:120] for c in row) + " |")
    if len(rows) > limit:
        lines.append(f"\n({len(rows)} rows returned; first {limit} shown, scoring used all {len(rows)})")
    return "\n".join(lines)

def exp_table(rows, limit=5):
    if not rows:
        return "(no rows)"
    cols = list(rows[0].keys())
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for r in rows[:limit]:
        lines.append("| " + " | ".join(str(r[c])[:80] for c in cols) + " |")
    return "\n".join(lines)

now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
slips = [r for r in results if r["slip_pct_finding"]]
md_out = []
md_out.append("# Genie Space Q&A transcript\n")
md_out.append(f"Space **Blue Ridge LNG — PM Co-Pilot (synthetic)** (`{SPACE_ID}`), asked through the Genie Conversation API from a one-time serverless job run (run id `{RUN_ID}`) as the submitting user, {now}. Data is synthetic and anchored to as-of date 2026-11-06.\n")
md_out.append("## Summary\n")
md_out.append(f"* **Curated: {cur['correct']}/{nc} correct** ({cur['partial']} partial, {cur['wrong']} wrong). Pass bar: 8/10.")
md_out.append(f"* **Held-out: {held['correct']}/{nh} correct** ({held['partial']} partial, {held['wrong']} wrong). Pass bar: 3/5. These five questions are not in the space.")
md_out.append(f"* Result: **{'PASS' if cur['correct'] >= 8 and held['correct'] >= 3 else 'BELOW THE PASS BAR'}**.")
md_out.append(f"* Answers that present slip_probability as a percentage or chance: **{len(slips)}**" + (" (" + ", ".join(f"{r['kind']} {r['n']}" for r in slips) + ")" if slips else "") + ".")
md_out.append("* Genie is not deterministic: another run can produce different SQL or wording, so a score is one sample, not a guarantee. Each question is a fresh conversation and none is retried.")
md_out.append("* Scoring is deterministic (key facts from the verified SQL found in Genie's answer or result rows; the rule is shown per question). Verified SQL and expected answers: `07_genie_agent/sample_questions.md` and `heldout_questions.md`.\n")
md_out.append("| # | Set | Question | Verdict | Facts found |")
md_out.append("|---|---|---|---|---|")
for r in results:
    tot = len(r["found"]) + len(r["missing"])
    md_out.append(f"| {r['n']} | {r['kind']} | {r['question']} | {r['verdict']} | {len(r['found'])}/{tot} |")
md_out.append("")
for kind, title in (("curated", "Curated questions (loaded into the space as example SQL)"), ("heldout", "Held-out questions (NOT in the space)")):
    md_out.append(f"## {title}\n")
    for r in [x for x in results if x["kind"] == kind]:
        md_out.append(f"### {kind} {r['n']}. {r['question']}\n")
        md_out.append(f"**Genie answer** ({r['seconds']}s, status {r['status']})\n")
        md_out.append("> " + (r["text"] or "(no text answer)").replace("\n", "\n> ") + "\n")
        md_out.append("**Generated SQL**\n")
        md_out.append("```sql\n" + (r["sql"] or "-- none") + "\n```\n")
        md_out.append("**Result rows**\n")
        md_out.append(table(r["columns"], r["rows"]) + "\n")
        md_out.append(f"**Expected answer:** {r['expected_answer']}\n")
        md_out.append("**Expected result (verified SQL)**\n")
        md_out.append(exp_table(r["expected_rows"]) + "\n")
        md_out.append(f"**Verdict: {r['verdict'].upper()}.** Rule: {r['rule_desc']}. Facts found: {r['found']}; missing: {r['missing']}." + ("".join(f" {n}." for n in r["notes"])) + "\n")
transcript = "\n".join(md_out)

os.makedirs("outputs", exist_ok=True)
with open("outputs/genie_qa_transcript.md", "w") as f:
    f.write(transcript)
with open("outputs/genie_qa_results.json", "w") as f:
    json.dump({"space_id": SPACE_ID, "run_id": RUN_ID, "generated": now, "results": results}, f, indent=1, ensure_ascii=False, default=str)
print(transcript)

# Genie Space Q&A transcript

Space **Blue Ridge LNG — PM Co-Pilot (synthetic)** (`01f1db9edffa1a298195db2806ce45e3`), asked through the Genie Conversation API from a one-time serverless job run (run id `770421906253489`) as the submitting user, 2026-11-08 18:13 UTC. Data is synthetic and anchored to as-of date 2026-11-06.

## Summary

* **Curated: 10/10 correct** (0 partial, 0 wrong). Pass bar: 8/10.
* **Held-out: 5/5 correct** (0 partial, 0 wrong). Pass bar: 3/5. These five questions are not in the space.
* Result: **PASS**.
* Answers that present slip_probability as a percentage or chance: **0**.
* Genie is not deterministic: another run can produce different SQL or wording, so a score is one sample, not a guarantee. Each question is a fresh conversation and none is retried.
* Scoring is deterministic (key facts from the verified SQL found in Genie's answer or result rows; the rule is shown per question). Verified SQL and expected answers: `07_genie_agent/sample_questions.md` and `h